In [2]:
import zipfile
import shutil
import tempfile
from pathlib import Path
from google.colab import files


def rename_pdf(filename, prefixes, skip_epi):
    stem = filename.stem

    if skip_epi and stem.startswith("EPI"):
        return filename.name

    for old, new in prefixes:
        if stem.startswith(old):
            return f"{new}{stem[len(old):]}{filename.suffix}"

    return filename.name


def rename_in_zip(zip_path, prefix_mapping, output_path):

    prefixes = sorted(
        prefix_mapping.items(),
        key=lambda x: len(x[0]),
        reverse=True
    )

    with tempfile.TemporaryDirectory() as tmp:

        tmp = Path(tmp)

        with zipfile.ZipFile(zip_path) as z:
            z.extractall(tmp)

        # Un solo recorrido
        pdfs = []

        has_epi = False
        has_hev = False

        for file in tmp.rglob("*.pdf"):

            pdfs.append(file)

            stem = file.stem

            if stem.startswith("EPI"):
                has_epi = True

            elif stem.startswith("HEV"):
                has_hev = True

        skip_epi = has_epi and has_hev

        for pdf in pdfs:

            new_name = rename_pdf(
                pdf,
                prefixes,
                skip_epi
            )

            if new_name != pdf.name:

                pdf.rename(
                    pdf.with_name(new_name)
                )

        output_zip = (
            output_path
            if output_path.lower().endswith(".zip")
            else f"{output_path}.zip"
        )

        shutil.make_archive(
            output_zip[:-4],
            "zip",
            tmp
        )

        print(f"✅ ZIP guardado como: {output_zip}")


# ---------------------------------------------------

print("=" * 60)
print("SUBA LOS ARCHIVOS ZIP")
print("=" * 60)

uploaded = files.upload()

prefix_mapping = {
    "SI": "FEV_900472731_SI",
    #"EPI": "HEV",
    #"OGR": "OPF",
    #"OPF": "PDE"
}

for fname in uploaded:

    rename_in_zip(
        fname,
        prefix_mapping,
        fname
    )

    files.download(fname)

SUBA LOS ARCHIVOS ZIP


Saving SI29797.zip to SI29797.zip
Saving SI29799.zip to SI29799.zip
Saving SI29801.zip to SI29801.zip
✅ ZIP guardado como: SI29797.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ ZIP guardado como: SI29799.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ ZIP guardado como: SI29801.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>